In [ ]:
%pip install ipympl

In [ ]:
%pip install numba   # one-time

In [20]:
"""
FULL “NEMATIC-ROD” NOTEBOOK CELL
────────────────────────────────
•  Populates **every** lattice site separated by `spacing_px` with one rod
•  Adds ± `jitter_pc` % positional disorder
•  Computes 2-D FFT → SAXS; shows radial I(q) with peak at q = 2π/d
•  Calculates CPM intensity pixel-wise from same image
•  Interactive dashboard: sliders *and* numeric boxes
"""

import numpy as np, matplotlib.pyplot as plt
from scipy import ndimage, fft
from scipy.ndimage import gaussian_filter1d, gaussian_filter
import ipywidgets as wd
from IPython.display import display, clear_output
from numba import njit, prange
from functools import lru_cache
from scipy.special import j1   # Bessel J₁



In [ ]:
%matplotlib widget

MC mayer saupe for real-space


In [19]:
"""
Self-avoiding nematic rods – Monte-Carlo (Maier–Saupe)
──────────────────────────────────────────────────────
• N_rods       : number of rods
• R, L         : radius, length  (px)
• ε_MS         : Maier–Saupe coupling (positive favours alignment)
• T            : reduced temperature  (k_BT = 1 in these units)
• sweeps       : how many attempted moves per rod
The code prints acceptance stats and shows a live image every 50 sweeps.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ────────────────────────── parameters
W, H   = 512, 512        # image size
N_rods = 400
R, L   = 3, 60
ε_MS   = 2.0             # Maier–Saupe coupling (in kBT units)
T      = 1.0
sweeps = 600

# helper: draw one rod as a binary mask
def rod_pixels(cx, cy, θ):
    half = L / 2
    xs = np.linspace(-half, half, int(L*2)+1)
    cθ, sθ = np.cos(θ), np.sin(θ)
    pix = []
    for t in xs:
        x = cx + t*cθ
        y = cy + t*sθ
        ix, iy = int(round(x)) % W, int(round(y)) % H
        for dx in range(-R, R+1):
            for dy in range(-R, R+1):
                if dx*dx + dy*dy <= R*R:
                    pix.append(((iy+dy) % H, (ix+dx) % W))
    return pix

# initialise rods random & non-overlapping
rods = []
occupied = set()
rng = np.random.default_rng(0)
while len(rods) < N_rods:
    cx, cy = rng.uniform(0, W), rng.uniform(0, H)
    θ = rng.uniform(0, 2*np.pi)
    pix = rod_pixels(cx, cy, θ)
    if not any(p in occupied for p in pix):      # hard-core
        rods.append([cx, cy, θ])
        occupied.update(pix)

# energy functions
def orient_energy(θ):
    return -ε_MS * 0.5 * (3*np.cos(θ)**2 - 1)    # −ε P₂(cosθ)

def total_energy():
    return sum(orient_energy(r[2]) for r in rods)

# Monte-Carlo
E = total_energy()
β = 1/T
disp = 6.0          # max translation per move
dθ   = 0.3          # max rotation per move
accept = move_try = 0

fig, ax = plt.subplots(figsize=(5,5))
for sweep in range(1, sweeps+1):
    for i in range(N_rods):
        cx, cy, θ = rods[i]
        move_try += 1
        # propose
        cx_new = (cx + rng.uniform(-disp, disp)) % W
        cy_new = (cy + rng.uniform(-disp, disp)) % H
        θ_new  = (θ  + rng.uniform(-dθ,  dθ)) % (2*np.pi)
        pix_old = rod_pixels(cx, cy, θ)
        pix_new = rod_pixels(cx_new, cy_new, θ_new)
        # overlap check
        if any(p in occupied and p not in pix_old for p in pix_new):
            continue
        # ΔE
        dE = orient_energy(θ_new) - orient_energy(θ)
        if dE <= 0 or rng.random() < np.exp(-β*dE):
            # accept
            accept += 1
            for p in pix_old: occupied.discard(p)
            for p in pix_new: occupied.add(p)
            rods[i] = [cx_new, cy_new, θ_new]
            E += dE
    # visualise every 50 sweeps
    if sweep % 50 == 0 or sweep == sweeps:
        ax.clear(); ax.set_aspect('equal')
        ax.set_xlim(0,W); ax.set_ylim(0,H)
        ax.set_xticks([]); ax.set_yticks([])
        for cx,cy,θ in rods:
            x0 = cx - (L/2)*np.cos(θ); y0 = cy - (L/2)*np.sin(θ)
            rect = patches.FancyArrow(x0, y0,
                                      L*np.cos(θ), L*np.sin(θ),
                                      width=2*R, head_width=0,
                                      color='k', length_includes_head=True)
            ax.add_patch(rect)
        ax.set_title(f"sweep {sweep}  acc {accept/move_try*100:.1f}%")
        plt.pause(0.001)

print(f"done: acceptance {accept/move_try:.3f}")

KeyboardInterrupt: 

In [ ]:
# ─── A. fast rod pixel painter ──────────────────────────────────
@njit
def _draw_rod_fast(img, θmap, cx, cy, θ, L, R):
    half = L / 2
    s    = np.linspace(-half, half, int(2*L)+1)
    cθ, sθ = np.cos(θ), np.sin(θ)
    H, W   = img.shape
    for t in s:
        x = cx + t*cθ
        y = cy + t*sθ
        ix = int(round(x)) % W
        iy = int(round(y)) % H
        for dx in range(-R, R+1):
            for dy in range(-R, R+1):
                if dx*dx + dy*dy <= R*R:
                    img[(iy+dy) % H, (ix+dx) % W] = 1
                    θmap[(iy+dy) % H, (ix+dx) % W] = θ

# call draw_rod_fast(...) inside make_sample() instead of _draw_rod
@lru_cache(maxsize=8)
def single_rod_fft(N, R, L, psf_sigma):
    """
    Returns |F(q)|² (normalised) of one rod centred in an NxN box,
    blurred with the same PSF sigma (px) as the sample.
    """
    rod = np.zeros((N, N))
    θmap = np.full((N, N), np.nan)
    _draw_rod(rod, θmap, N/2, N/2, 0.0, L, R)   # draw along +x
    F = fft.fftshift(fft.fft2(rod))
    I = np.abs(F)**2
    if psf_sigma > 0:
        I = gaussian_filter(I, sigma=psf_sigma, mode='nearest')
    return I / I.max()
# ───── Helper to draw one rod
def _draw_rod(img, θmap, cx, cy, θ, L, R):
    half=L/2
    s=np.linspace(-half, half, int(2*L)+1)
    for x,y in zip(cx+s*np.cos(θ), cy+s*np.sin(θ)):
        ix,iy=int(round(x))%img.shape[1], int(round(y))%img.shape[0]
        for dx in range(-R,R+1):
            for dy in range(-R,R+1):
                if dx*dx+dy*dy<=R*R:
                    img[(iy+dy)%img.shape[0], (ix+dx)%img.shape[1]] = 1
                    θmap[(iy+dy)%img.shape[0], (ix+dx)%img.shape[1]] = θ

# ───── Build full lattice
def make_sample(N,R,L,α,Δα,spacing,jitter):
    ρ=np.zeros((N,N)); θ=np.full((N,N),np.nan)
    xs=np.arange(spacing/2,N,spacing); ys=xs
    coords=np.stack(np.meshgrid(xs,ys),-1).reshape(-1,2)
    coords+=np.random.uniform(-jitter/100*spacing, jitter/100*spacing, coords.shape)
    for cx,cy in coords:
       # _draw_rod(ρ,θ,cx,cy,np.deg2rad(α+np.random.normal(0,Δα)),L,R)
       _draw_rod_fast(ρ,θ,cx,cy,np.deg2rad(α+np.random.normal(0,Δα)),L,R)

    return ρ,θ


def compute_fft(im, im_min=0.004, kill_center=False):
    im_norm = (im.astype(float) - im.min()) / (im.max() - im.min())
    im_norm = np.maximum(im_norm, im_min)
    if im_norm.ndim == 3 and im_norm.shape[2] > 1:
        intensity = np.zeros_like(im_norm)
        for ci in range(im_norm.shape[2]):
            f = np.fft.fft2(im_norm[:, :, ci])
            fshift = np.fft.fftshift(f)
            intensity[:, :, ci] = np.abs(fshift) ** 2
    else:
        f = np.fft.fft2(im_norm)
        fshift = np.fft.fftshift(f)
        intensity = np.abs(fshift) ** 2
    intensity += im_min
    if kill_center:
        h, w = intensity.shape[:2]
        if intensity.ndim == 2:
            intensity[h//2-1:h//2+1, :] = intensity[h//2+1:h//2+3, :]
            intensity[:, w//2-1:w//2+1] = intensity[:, w//2+1:w//2+3]
        else:
            intensity[h//2-1:h//2+1, :, :] = intensity[h//2+1:h//2+3, :, :]
            intensity[:, w//2-1:w//2+1, :] = intensity[:, w//2+1:w//2+3, :]
    return im_norm, intensity

# Radial integration

def radial_profile(data):
    y, x = np.indices(data.shape)
    center = np.array([(x.max() - x.min())/2.0,
                       (y.max() - y.min())/2.0])
    r = np.hypot(x - center[0], y - center[1])
    r_int = r.astype(np.int32)
    max_r = int(min(center))
    tbin = np.bincount(r_int.ravel(), data.ravel(), minlength=max_r+1)
    nr = np.bincount(r_int.ravel(), minlength=max_r+1)
    return tbin[:max_r+1] / nr[:max_r+1]


# ──────────────────────────────────────────
# SAXS analysis
# returns I2 (blurred total intensity), radial tuple, azimuth tuple
# ──────────────────────────────────────────
def saxs(ρ, spacing, band_mult, psf_sigma, R, L, smooth_sigma=1.5):
    """
    spacing     : centre-to-centre lattice spacing (px)
    band_mult   : annulus half-width in Δq units for χ histogram
    psf_sigma   : Gaussian blur σ (px) that mimics beam PSF
    R, L        : rod radius & length (px)  –– used only for q-grid size
    """
    N = ρ.shape[0]

    # 1. FFT → I(q)
    # F  = fft.fftshift(fft.fft2(ρ))
    # I2 = np.abs(F)**2
    # I2 /= I2.max()

    # # 2. mask qx=0 & qy=0 axes, 3×3 mean fill
    # # -- mask only the very centre to kill DC spike ------------------
    # mid = N // 2
    # I2[mid-1:mid+2, mid-1:mid+2] = np.nan      # 3×3 NaNs at origin

    im_norm, I2 = compute_fft(ρ, kill_center=True)
    # -- PSF blur (optional) -----------------------------------------
    if psf_sigma > 0:
        I2 = gaussian_filter(I2, sigma=psf_sigma, mode='nearest')

    # -- safe normalisation ------------------------------------------
    finite_max = np.nanmax(I2)
    I2 /= finite_max if finite_max > 0 else 1.0

    # ----- keep this copy for radial plotting -----
    I2_radial = I2.copy()

    # 4. coordinate grids
    k  = fft.fftshift(fft.fftfreq(N)) * 2*np.pi
    Kx, Ky = np.meshgrid(k, k, indexing="ij")
    K  = np.hypot(Kx, Ky)
    χ  = (np.rad2deg(np.arctan2(Ky, Kx)) + 360) % 360   # 0–360 °

    # 5. radial bins
    q_bins = np.linspace(0, K.max(), N + 1)         # N annuli
    bin_id = np.digitize(K.ravel(), q_bins) - 1

    # ---- radial profile from total intensity (structure × FF) ----
    I_sum = ndimage.sum(I2_radial.ravel(), labels=bin_id,
                        index=np.arange(N))
    I_sum[0] = 0
    I_r = I_sum / (I_sum.max() + 1e-12)
    q_mid = 0.5 * (q_bins[:-1] + q_bins[1:])

    q_th  = 2*np.pi / spacing
    pk    = np.argmin(np.abs(q_mid - q_th))
    q_pk  = q_mid[pk]
    Δq    = band_mult * (q_bins[1] - q_bins[0])

    # 6. flatten rod form factor → S(q) for azimuth
    bg = ndimage.mean(I2.ravel(), labels=bin_id, index=np.arange(N))
    bg[bg == 0] = 1e-12
    I2f = I2 / bg[np.clip(np.digitize(K, q_bins) - 1, 0, N - 1)]

    # 7. azimuth χ histogram in q ± Δq
    mask = (K > q_pk - Δq) & (K < q_pk + Δq)
    χ_bins = np.linspace(0, 360, 361)
    Iχ, _  = np.histogram(χ[mask], χ_bins, weights=I2f[mask])
    Iχ = gaussian_filter1d(Iχ, sigma=smooth_sigma)
    Iχ /= Iχ.max() + 1e-12
    χ_mid = 0.5 * (χ_bins[:-1] + χ_bins[1:])
    
    
    
    I2f = I2 / bg[np.clip(np.digitize(K, q_bins)-1, 0, N-1)]   # structure factor
    S2  = I2f                                                  # ← alias

    return I2, S2, (q_mid, I_r, q_pk, q_th, Δq), (χ_mid, Iχ)

# ──────────────────────────────────────────
# Dashboard callback
# ──────────────────────────────────────────
def run(N, R, L, α, Δα, spacing, jitter, band, psf):
    clear_output(wait=True)

    # make sample & SAXS
    ρ, θmap = make_sample(N, R, L, α, Δα, spacing, jitter)
    I2, S2, (q, I_r, q_pk, q_th, Δq), (χ, Iχ) = saxs(
    ρ, spacing, band, psf_sigma=psf, R=R, L=L
    )
    φ = np.linspace(0, 360, 721)
    Iφ = cpm_curve(θmap, φ)

    # figure grid  (3×2)
    fig = plt.figure(figsize=(18, 9))
    fig.canvas.toolbar_visible = True
    fig.canvas.header_visible  = False
    gs  = fig.add_gridspec(2, 3, width_ratios=[1.3, 1.3, 1.1])

    # 1. real-space image
    ax = fig.add_subplot(gs[0, 0])
    ax.imshow(ρ, cmap='gray', origin='lower')
    ax.set_title("Real space"); ax.axis('off')

    # 2. SAXS 2-D with rings
    ax_fft = fig.add_subplot(gs[0, 1])
    ax_fft.imshow(np.log10(I2 + 1e-6), origin='lower', cmap='viridis')
    ax_fft.set_title(f"SAXS log₁₀  (PSF σ={psf})"); ax_fft.axis('off')
    pix_per_rad = N / (2*np.pi)
    for r, col in [(pix_per_rad*(q_pk-Δq), 'cyan'),
                   (pix_per_rad*q_pk,       'red'),
                   (pix_per_rad*(q_pk+Δq), 'cyan')]:
        circ = plt.Circle((N/2, N/2), r, fill=False,
                          edgecolor=col, linewidth=1.3)
        ax_fft.add_patch(circ)

    # 3. radial I(q) + analytical form factor
    ax = fig.add_subplot(gs[0, 2])
    ax.plot(q, I_r, lw=1.8, label='Structure × FF')
    # cylinder orientation-averaged P(q)
    q_safe = np.where(q == 0, 1e-12, q)
    Fq = (2 * j1(q_safe * R) / (q_safe * R)) * np.sinc(q_safe * L / (2*np.pi))
    Pq = (Fq**2) / (Fq**2).max()
    ax.plot(q, Pq, color='red', lw=1.4, label='Single-rod FF')
    ax.axvline(q_pk, c='r', ls='--', alpha=0.5, label=f'q_peak={q_pk:.3f}')
    ax.axvline(q_th, c='g', ls=':',  alpha=0.5, label=r'$2\pi/d$')
    ax.set_xlabel('q (rad px⁻¹)'); ax.set_ylabel('Normalised I(q)')
    ax.set_title('Radial profile'); ax.legend(fontsize=8)
    
    # ------------ q-line cut  (structure factor S2) ------------------
    stripe_px = 4                       # half-width in qx pixels
    mid = S2.shape[1] // 2
    band = S2[:, mid-stripe_px : mid + stripe_px + 1]   # average over 2·stripe+1 cols
    I1d = band.mean(axis=1)
    qy  = (np.arange(S2.shape[0]) - S2.shape[0]/2) * (2*np.pi / S2.shape[0])

    mask = qy >= 0
    qy, I1d = qy[mask], I1d[mask] / I1d[mask].max()

    ax_line = fig.add_subplot(gs[1, 2])
    ax_line.plot(qy, I1d, lw=1.2)
    for n in range(1, 6):
        ax_line.axvline(n * 2*np.pi / spacing, ls=':', c='k',
                        alpha=0.6, label='_nolegend_')
    ax_line.set_xlim(0, 5 * 2*np.pi / spacing)
    ax_line.set_xlabel(r"$q_y$  (rad px$^{-1}$)")
    ax_line.set_ylabel("Normalised S(q)")
    ax_line.set_title("Lamellar harmonics (qx≈0)")
    
    # 4. azimuth χ
    ax = fig.add_subplot(gs[1, 0])
    ax.plot(χ, Iχ)
    ax.set_xlim(0, 360); ax.set_xlabel('χ (deg)'); ax.set_ylabel('Iχ')
    ax.set_title(f"Azimuth  q±{band}Δq  (Δq≈{Δq:.3f})")

    # 5. CPM vs SAXS
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(φ, Iφ, label='CPM'); ax.plot(χ, Iχ, label='SAXS')
    ax.set_xlim(0, 360); ax.set_ylim(0, 1.05); ax.grid(alpha=0.4)
    ax.set_xlabel('Angle vs +x (deg)'); ax.set_ylabel('Normalised I')
    ax.set_title('CPM vs SAXS'); ax.legend()

    # 6. reference axes
    axr = fig.add_subplot(gs[1, 2], aspect='equal'); axr.axis('off')
    axr.set_xlim(-1.1, 1.1); axr.set_ylim(-1.1, 1.1)
    axr.set_title('Reference')
    axr.arrow(0, 0, 1, 0, color='lime', width=0.03, head_width=0.07)
    axr.text(1.05, 0, 'φ=0°', va='center', color='lime')
    dx, dy = np.cos(np.deg2rad(α)), np.sin(np.deg2rad(α))
    axr.arrow(-dx*0.5, -dy*0.5, dx, dy, color='orange',
              width=0.03, head_width=0.07)
    axr.arrow(dx*0.5, dy*0.5, -dx, -dy, color='orange',
              width=0.03, head_width=0.07)
    axr.text(dx*0.6, dy*0.6, f'α={α}°', color='orange')

    fig.tight_layout(); plt.show()

# ───── Widget factory ────────────────────────────────────────────


def slider_pair(label, val, mn, mx, step, flt=False):
    """
    Returns (HBox, slider_widget).
    * If flt=True  → FloatSlider + BoundedFloatText
    * If flt=False → IntSlider   + BoundedIntText
    """
    Slider = wd.FloatSlider if flt else wd.IntSlider
    Text   = wd.BoundedFloatText if flt else wd.BoundedIntText

    s = Slider(value=val, min=mn, max=mx, step=step,
               description=label, continuous_update=False)
    t = Text(value=val, min=mn, max=mx, step=step)
    wd.jslink((s, 'value'), (t, 'value'))
    return wd.HBox([s, t]), s


# ── create sliders ────────────────────────────────────────────────
bxN,   wN   = slider_pair('Grid N',     1024, 512, 2048, 256)
bxR,   wR   = slider_pair('Radius',        3,   1,    10,   1)
bxL,   wL   = slider_pair('Length',       64,  16,   160,   8)
bxA,   wA   = slider_pair('α°',           30,   0,   179,   1)
bxDA,  wDA  = slider_pair('Δα°',           5,   0,    40,   1)
bxSp,  wSp  = slider_pair('Spacing',      32,   8,   128,   4)
bxJ,   wJ   = slider_pair('Jitter %',      5,   0,    30, 0.5, flt=True)
bxB,   wB   = slider_pair('band×Δq',       3, 0.5,     50, 0.5, flt=True)
bxPSF, wPSF = slider_pair('PSF σ (px)',    2, 0.0,    50, 0.2, flt=True)

# ── lay out the control column ────────────────────────────────────
ui = wd.VBox([
    bxN, bxR, bxL,         # first three sliders
    bxA, bxDA, bxSp,       # orientation & spacing
    bxJ, bxB, bxPSF        # jitter, Δq band, PSF blur
])

# ── connect sliders to the dashboard callback `run()` ─────────────
out = wd.interactive_output(
    run,
    {
        'N'      : wN,
        'R'      : wR,
        'L'      : wL,
        'α'      : wA,
        'Δα'     : wDA,
        'spacing': wSp,
        'jitter' : wJ,
        'band'   : wB,
        'psf'    : wPSF
    }
)

# ── show everything ───────────────────────────────────────────────
display(ui, out)

Output()